# ListenBrainz Data Cleaning

**Student:** Jingxuan (Jennifer) Zhong  
**Project:** Emerging Artist Discovery Through Listening Behavior  
**Individual branch:** cleaning-jennifer

## Project Vision and Question

A music-streaming editorial team wants to identify emerging artists with small initial audiences and sustained listener growth.

**Question:** Which artists show growth in unique listeners and strong repeat engagement during the observation period? The project measures unique listeners, listener growth, repeat listens, distinct listening days, and distinct tracks per user and artist.

One row is one user's listen to one track at one point in time.

## Load the Dataset

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
import clean_listenbrainz as cleaner

source_path = Path("data/raw/listenbrainz_sample.csv")
raw = cleaner.load_source(source_path)
print(f"Rows: {len(raw):,}; columns: {raw.shape[1]}")
print("Columns:", raw.columns.tolist())
display(raw.head(6))

Rows: 1,940; columns: 9
Columns: ['username', 'listened_at', 'track_name', 'artist_name', 'release_name', 'recording_msid', 'artist_mbids', 'recording_mbid', 'release_mbid']


,username,listened_at,track_name,artist_name,release_name,recording_msid,artist_mbids,recording_mbid,release_mbid
0,iliekcomputers,2024-12-16 08:04:37,Take Care,"Drake, Rihanna",Take Care (Deluxe),a9dccce2-eb7b-4e87-ba43-0cf9219997b7,"['9fff2f8a-21e6-47de-a2b8-7f449929d43f', '73e5...",a23ccdb8-765f-4b7d-b47a-63191b2f856f,02f7e348-b1ae-4c4e-a057-b97f1f56e53c
1,iliekcomputers,2024-12-16 05:56:01,Us Against The World,Jungle,Volcano,c97b7e4d-17f9-4a1b-a26e-ec107bae5db0,['6bbb3983-ce8a-4971-96e0-7cae73268fc4'],14f9f386-3df5-4be3-b92c-e473b0afd98e,a041661f-d1c4-4398-b23d-6bd156bf1aed
2,iliekcomputers,2024-12-16 05:44:12,Love on the Ground,Cannons,Shadows,000c52a6-b412-4a92-a948-ce6fc35bd2e6,['eb9eb9f8-c691-4fef-9bdf-0cbffbfa42ee'],2b49aa61-3d97-4555-811c-204f56d0a0fa,f9ed6b4a-3357-4f0a-907a-5f73cea0b72e
3,iliekcomputers,2024-12-16 05:40:30,Shadows,Cannons,Shadows,6ddd1875-d288-40f8-ace6-8e3b598eaa59,['eb9eb9f8-c691-4fef-9bdf-0cbffbfa42ee'],16279f1d-ddc5-455c-b3db-b70a094af533,f9ed6b4a-3357-4f0a-907a-5f73cea0b72e
4,iliekcomputers,2024-12-16 05:36:34,Bright Lights,Cannons,Shadows,769d59c9-be4f-4097-8607-ba8deb018321,['eb9eb9f8-c691-4fef-9bdf-0cbffbfa42ee'],b0f87b27-ccfe-423a-8428-8af152211fc4,f9ed6b4a-3357-4f0a-907a-5f73cea0b72e
5,iliekcomputers,2024-12-16 05:33:08,Love Chained,Cannons,Shadows,6c0ccb7b-6b0e-4798-a713-794390232366,['eb9eb9f8-c691-4fef-9bdf-0cbffbfa42ee'],d8db1d36-9819-41bc-8ad7-e67a4ec167ed,f9ed6b4a-3357-4f0a-907a-5f73cea0b72e


## Understand the Data

In [2]:
display(pd.DataFrame({
    "data_type": raw.dtypes.astype(str),
    "non_empty_values": raw.ne("").sum(),
    "unique_values": raw.nunique(dropna=True),
}))
dates = pd.to_datetime(raw["listened_at"], errors="coerce", utc=True)
date_failures = raw["listened_at"].ne("") & dates.isna()
as_of = pd.Timestamp.now(tz="UTC")
future_dates = dates.notna() & dates.gt(as_of)
print("Timestamp examples:", raw["listened_at"].head(4).tolist())
print("Conversion failures:", int(date_failures.sum()))
print("Valid date range:", dates.min(), "to", dates.max())
print("Future dates:", int(future_dates.sum()))
print("Unique users, artists, tracks, MSIDs, and recording MBIDs:")
display(raw[["username", "artist_name", "track_name", "recording_msid", "recording_mbid"]]
        .replace("", pd.NA).nunique().rename("unique_count").to_frame())

Timestamp examples: ['2024-12-16 08:04:37', '2024-12-16 05:56:01', '2024-12-16 05:44:12', '2024-12-16 05:40:30']
Conversion failures: 0
Valid date range: 2019-03-07 13:17:39+00:00 to 2026-09-21 03:51:02+00:00
Future dates: 0
Unique users, artists, tracks, MSIDs, and recording MBIDs:


,data_type,non_empty_values,unique_values
username,string,1940,5
listened_at,string,1940,1940
track_name,string,1940,1308
artist_name,string,1940,756
release_name,string,1186,576
recording_msid,string,1940,1330
artist_mbids,string,1181,564
recording_mbid,string,1212,873
release_mbid,string,1181,621


,unique_count
username,5
artist_name,756
track_name,1308
recording_msid,1330
recording_mbid,872


## Missing Values

In [3]:
pandas_na_view = pd.read_csv(source_path)
text_columns = raw.columns
blank_counts = raw.eq("").sum()
whitespace_only = raw.apply(lambda col: col.str.fullmatch(r"\\s*", na=False).sum())
tokens = {"NA", "N/A", "null", "None", "unknown"}
placeholder_counts = raw.apply(
    lambda col: col.str.strip().str.casefold().isin({token.casefold() for token in tokens}).sum()
)
missing_report = pd.DataFrame({
    "pandas_missing": pandas_na_view.isna().sum(),
    "blank_fields": blank_counts,
    "whitespace_only": whitespace_only,
    "placeholder_tokens": placeholder_counts,
})
display(missing_report)

,pandas_missing,blank_fields,whitespace_only,placeholder_tokens
username,0,0,0,0
listened_at,0,0,0,0
track_name,0,0,0,0
artist_name,0,0,0,2
release_name,754,754,0,0
recording_msid,0,0,0,0
artist_mbids,759,759,0,0
recording_mbid,728,728,0,0
release_mbid,759,759,0,0


## Duplicate Records

In [4]:
print("Exact duplicate rows:", int(raw.duplicated().sum()))
key_rows = []
for key in cleaner.CANDIDATE_KEYS:
    complete = cleaner.key_complete(raw, key)
    values = raw.loc[complete, key]
    key_rows.append({
        "candidate_key": " + ".join(key),
        "duplicate_occurrences": int(values.duplicated().sum()),
        "rows_in_repeated_keys": int(values.duplicated(keep=False).sum()),
        "rows_with_missing_key_parts": int((~complete).sum()),
    })
display(pd.DataFrame(key_rows))

Exact duplicate rows: 0


,candidate_key,duplicate_occurrences,rows_in_repeated_keys,rows_with_missing_key_parts
0,username + listened_at + recording_msid,0,0,0
1,username + listened_at + recording_mbid,0,0,728
2,username + listened_at + artist_name + track_name,0,0,0


## Data Cleaning

The rules below keep listen events, retain missing metadata, convert listening times to UTC, trim edge whitespace, and flag unusual dates or identifiers. Artist names equal to Unknown are treated as missing. Artist MBIDs remain in the event table; multiple credits are not reduced to one artist.

In [5]:
decision_log = pd.DataFrame([
    {"rule": "Keep every listen; flag exact duplicates and candidate-key repeats.",
     "reason": "Repeated listening can be real behavior.",
     "check": "Compare row counts and duplicate/key checks."},
    {"rule": "Convert listened_at to UTC; keep valid values and flag invalid or future dates.",
     "reason": "Time-based growth and listening-day measures need consistent timestamps.",
     "check": "Count parse failures and future dates; compare source/output rows."},
    {"rule": "Do not fill or drop missing identifiers or metadata.",
     "reason": "Unresolved metadata should remain unknown.",
     "check": "Compare missing counts and inspect blank output fields."},
    {"rule": "Treat artist_name Unknown as missing; keep other text values.",
     "reason": "Unknown does not identify an artist.",
     "check": "Compare Unknown values before and after cleaning."},
    {"rule": "Trim leading/trailing whitespace only.",
     "reason": "Edge spaces can affect grouping; internal formatting can be meaningful.",
     "check": "Count whitespace before and after trimming."},
    {"rule": "Keep MBID/MSID columns as text and flag invalid UUID-style identifiers.",
     "reason": "Identifiers are labels and invalid values need review.",
     "check": "Validate UUID-style format without removing rows."},
    {"rule": "Use a valid single artist MBID as the row identity; use a name fallback only when no MBID is present.",
     "reason": "A name fallback is not the same as a verified MBID; multi-artist credits cannot be represented by one key.",
     "check": "Review identity-key types and preserve the original artist_mbids field."},
])
display(decision_log)

,rule,reason,check
0,Keep every listen; flag exact duplicates and c...,Repeated listening can be real behavior.,Compare row counts and duplicate/key checks.
1,Convert listened_at to UTC; keep valid values ...,Time-based growth and listening-day measures n...,Count parse failures and future dates; compare...
2,Do not fill or drop missing identifiers or met...,Unresolved metadata should remain unknown.,Compare missing counts and inspect blank outpu...
3,Treat artist_name Unknown as missing; keep oth...,Unknown does not identify an artist.,Compare Unknown values before and after cleaning.
4,Trim leading/trailing whitespace only.,Edge spaces can affect grouping; internal form...,Count whitespace before and after trimming.
5,Keep MBID/MSID columns as text and flag invali...,Identifiers are labels and invalid values need...,Validate UUID-style format without removing rows.
6,Use a valid single artist MBID as the row iden...,A name fallback is not the same as a verified ...,Review identity-key types and preserve the ori...


In [6]:
cleaned = cleaner.clean_data(raw)
cleaned_sample = cleaner.select_cleaned_sample(raw, cleaned, max_rows=50)
full_path = Path("data/processed/listenbrainz_cleaned.csv")
sample_path = Path("cleaned_sample.csv")
cleaner.write_outputs(cleaned, cleaned_sample, full_path, sample_path)
print(f"Cleaned rows: {len(cleaned):,}")
print(f"Sample rows: {len(cleaned_sample):,}")
print("Sample reasons:")
display(cleaned_sample["sample_reason"].value_counts().rename_axis("reason").to_frame("rows"))

Cleaned rows: 1,940
Sample rows: 50
Sample reasons:


,rows
reason,
Reproducible supplemental sample,22
Missing optional metadata preserved,10
Multiple artist MBIDs retained,8
Repeated recording listen retained,3
Name punctuation preserved,3
Unknown artist name recoded to missing,2
Multiple artist MBIDs retained; Name punctuation preserved; Repeated recording listen retained,1
Multiple artist MBIDs retained; Name punctuation preserved,1


## Validation

In [7]:
validation = cleaner.validate_outputs(raw, cleaned, cleaned_sample)
full_saved = pd.read_csv(full_path, dtype="string", keep_default_na=False)
sample_saved = pd.read_csv(sample_path, dtype="string", keep_default_na=False)
validation.loc[len(validation)] = ["saved full row count", len(raw), len(full_saved), len(full_saved) == len(raw)]
validation.loc[len(validation)] = ["saved sample maximum", 50, len(sample_saved), len(sample_saved) <= 50]
display(validation)
assert validation["passed"].all()
print("Invalid identifier flags:", {
    column: int(cleaned[column].sum())
    for column in cleaned.columns if column.endswith("_invalid_review")
})

Invalid identifier flags: {'recording_msid_invalid_review': 0, 'recording_mbid_invalid_review': 0, 'release_mbid_invalid_review': 0, 'artist_mbids_invalid_review': 0}


,check,expected,actual,passed
0,source rows retained,1940,1940,True
1,recording order retained,True,True,True
2,sample at most 50 rows,50,50,True
3,all difficult-case categories included,5,5,True
4,saved full row count,1940,1940,True
5,saved sample maximum,50,50,True


### Five Record Checks

In [8]:
checks = []
def add_check(label, source_index, expected, actual, passed):
    checks.append({
        "record": label,
        "source_row": int(source_index) + 2,
        "original": str(raw.loc[source_index].to_dict()),
        "expected": str(expected),
        "actual": str(actual),
        "passed": bool(passed),
    })

unknown_index = raw.index[raw["artist_name"].str.strip().str.casefold().eq("unknown")][0]
clean_unknown = cleaned.loc[cleaned["source_row_number"].eq(unknown_index + 2)].iloc[0]
add_check("Unknown artist becomes missing", unknown_index, "artist_name is blank; row retained",
          f"artist_name={clean_unknown['artist_name']!s}; row retained",
          pd.isna(clean_unknown["artist_name"]))

multi_index = raw.index[raw["artist_mbids"].map(lambda value: len(cleaner.parse_artist_mbids(value)) > 1)][0]
clean_multi = cleaned.loc[cleaned["source_row_number"].eq(multi_index + 2)].iloc[0]
add_check("Multiple artist MBIDs preserved", multi_index, raw.loc[multi_index, "artist_mbids"],
          clean_multi["artist_mbids"], raw.loc[multi_index, "artist_mbids"].strip() == clean_multi["artist_mbids"])

missing_index = raw.index[raw["artist_mbids"].eq("")][0]
clean_missing = cleaned.loc[cleaned["source_row_number"].eq(missing_index + 2)].iloc[0]
add_check("Missing artist metadata stays blank", missing_index, "artist_mbids remains blank; row retained",
          f"artist_mbids={clean_missing['artist_mbids']!r}; row retained",
          clean_missing["artist_mbids"] == "")

repeated_values = raw["recording_msid"][raw["recording_msid"].duplicated(keep=False)]
repeat_id = repeated_values.iloc[0]
repeat_indices = raw.index[raw["recording_msid"].eq(repeat_id)]
retained = int(cleaned["recording_msid"].eq(repeat_id).sum())
add_check("Repeated recording listens retained", repeat_indices[0],
          f"{len(repeat_indices)} events with the same MSID remain",
          f"{retained} events with the same MSID remain", retained == len(repeat_indices))

date_index = raw.index[raw["listened_at"].ne("")][0]
clean_date = cleaned.loc[cleaned["source_row_number"].eq(date_index + 2), "listened_at"].iloc[0]
date_passed = clean_date.tz is not None and str(clean_date.tz) == "UTC"
add_check("Listening time converted to UTC", date_index, "timezone-aware UTC timestamp",
          clean_date.isoformat(), date_passed)

record_checks = pd.DataFrame(checks)
display(record_checks[["record", "source_row", "original", "expected", "actual", "passed"]])
assert record_checks["passed"].all()

,record,source_row,original,expected,actual,passed
0,Unknown artist becomes missing,1447,"{'username': 's_john_s', 'listened_at': '2019-...",artist_name is blank; row retained,artist_name=<NA>; row retained,True
1,Multiple artist MBIDs preserved,2,"{'username': 'iliekcomputers', 'listened_at': ...","['9fff2f8a-21e6-47de-a2b8-7f449929d43f', '73e5...","['9fff2f8a-21e6-47de-a2b8-7f449929d43f', '73e5...",True
2,Missing artist metadata stays blank,33,"{'username': 'iliekcomputers', 'listened_at': ...",artist_mbids remains blank; row retained,artist_mbids=''; row retained,True
3,Repeated recording listens retained,2,"{'username': 'iliekcomputers', 'listened_at': ...",3 events with the same MSID remain,3 events with the same MSID remain,True
4,Listening time converted to UTC,2,"{'username': 'iliekcomputers', 'listened_at': ...",timezone-aware UTC timestamp,2024-12-16T08:04:37+00:00,True


## Summary

In [9]:
print(f"Source and cleaned event rows: {len(raw):,} and {len(cleaned):,}")
print("Artist names recoded to missing:", int(raw["artist_name"].str.strip().str.casefold().eq("unknown").sum()))
print("Duplicate and candidate-key rows flagged:",
      int(cleaned["exact_duplicate_review"].sum()), int(cleaned["candidate_key_review"].sum()))
print("Invalid/future dates flagged:", int(cleaned["invalid_date_review"].sum()),
      int(cleaned["future_date_review"].sum()))
print("Rows in the submitted cleaned sample:", len(cleaned_sample))
print("The source remains unchanged; the full cleaned file is kept locally for validation.")

Source and cleaned event rows: 1,940 and 1,940
Artist names recoded to missing: 2
Duplicate and candidate-key rows flagged: 0 0
Invalid/future dates flagged: 0 0
Rows in the submitted cleaned sample: 50
The source remains unchanged; the full cleaned file is kept locally for validation.
